# Immobilier et performance énergétique en France

Projet du cours Python pour la data science, ENSAE Paris.
Marc Olivier Bouiti Banza, Karlane Vasthi Foudjin Sipeuwo, Aniela Nomwendé Tania Zoungrana.

Ce notebook est le rapport du projet. Il s'exécute du début à la fin à partir des fichiers produits par la chaîne de traitement (voir le README).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))
from src import chargement, config

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

ORDRE = chargement.ETIQUETTES
COULEURS = {"A": "#009C6D", "B": "#52B153", "C": "#78BD76", "D": "#F4E70F",
            "E": "#F0B40F", "F": "#EB8235", "G": "#D7221F"}

## Introduction

*À rédiger.* La question, le contexte (réforme du DPE de juillet 2021, loi Climat et résilience, gel des loyers des logements F et G, interdiction de louer les logements G depuis 2025) et l'étude de référence (Chareyron, 2026).

## Partie 1. Les données

### 1.1 Sources et traitement

*À rédiger.* Les trois sources, la chaîne de traitement et les six règles d'appariement (détail en annexe A).

![Chaîne de traitement](../docs/chaine_de_traitement.png)

In [ ]:
ventes = chargement.charger_ventes()
echantillon = chargement.echantillon_analyse(ventes)
print(f"{len(ventes):,} ventes ; {ventes['apparie'].sum():,} appariées ; "
      f"{len(echantillon):,} dans l'échantillon d'analyse")

### 1.2 Le marché

Prix médian au m² de toutes les ventes, appariées ou non, par trimestre et par département.

In [ ]:
prix_trimestre = (ventes.groupby(["trimestre", "type_bien"], observed=True)["prix_m2"]
                  .median()
                  .unstack())

ax = prix_trimestre.plot(marker="o", color={"appartement": "#14213D", "maison": "#A23B3B"})
ax.set_xticks(range(len(prix_trimestre)), prix_trimestre.index, rotation=45, ha="right")
ax.set(title="Prix médian au m² par trimestre (toutes les ventes)", xlabel="", ylabel="€/m²")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
from matplotlib.colors import ListedColormap

# Contours des départements (projet france-geojson de Grégoire David)
URL_DEPARTEMENTS = ("https://raw.githubusercontent.com/gregoiredavid/france-geojson/"
                    "master/departements-version-simplifiee.geojson")
departements = gpd.read_file(URL_DEPARTEMENTS)

prix_dep = ventes.groupby("code_departement", observed=True)["prix_m2"].median().reset_index()
prix_dep["code_departement"] = prix_dep["code_departement"].astype(str)
carte = departements.merge(prix_dep, left_on="code", right_on="code_departement", how="left")

# Classes de prix : sinon Paris (plus de 10 000 €/m²) écraserait toutes les autres couleurs
CLASSES = [0, 1500, 2000, 2500, 3000, 4000, 20000]
NOMS = ["moins de 1 500", "1 500 à 2 000", "2 000 à 2 500", "2 500 à 3 000", "3 000 à 4 000", "plus de 4 000"]
carte["classe_prix"] = pd.cut(carte["prix_m2"], CLASSES, labels=NOMS)

bleus = ListedColormap(plt.cm.Blues(np.linspace(0.25, 1, len(NOMS))))
fig, ax = plt.subplots(figsize=(7, 7))
carte.plot(column="classe_prix", cmap=bleus, categorical=True, legend=True, ax=ax,
           edgecolor="white", linewidth=0.3,
           missing_kwds={"color": "lightgrey", "label": "hors champ"},
           legend_kwds={"title": "€/m²", "loc": "center left", "bbox_to_anchor": (1, 0.5), "frameon": False})
ax.set_title("Prix médian au m² par département, 2022 à 2025")
ax.set_axis_off()
plt.show()

En gris : la Moselle, le Bas-Rhin et le Haut-Rhin, absents de la base des ventes (régime du livre foncier).

*À rédiger.* Commentaire des deux graphiques.

### 1.3 L'échantillon

Part des ventes auxquelles un DPE a été rattaché, puis comparaison des ventes appariées et non appariées. Le détail par département est en annexe B.

In [ ]:
taux = (ventes.groupby(["annee", "type_bien"], observed=True)["apparie"]
        .mean()
        .mul(100)
        .unstack())
taux["ensemble"] = ventes.groupby("annee")["apparie"].mean() * 100
taux.round(1)

In [ ]:
statut = ventes["apparie"].map({True: "appariées", False: "non appariées"})

comparaison = (ventes.groupby(["annee", statut], observed=True)
               .agg(ventes=("prix_m2", "size"),
                    prix_m2_median=("prix_m2", "median"),
                    surface_mediane=("surface", "median"),
                    part_maisons=("type_bien", lambda x: (x == "maison").mean() * 100))
               .round(1))
comparaison.index.names = ["année", "statut"]
comparaison

*À rédiger.* Commentaire : évolution du taux d'appariement et différences entre ventes appariées et non appariées.

## Partie 2. Les résultats

Toutes les analyses de cette partie portent sur l'échantillon d'analyse : les ventes appariées, sans les ventes rapprochées.

### 2.1 Description par étiquette

In [ ]:
part_maisons = (echantillon["type_bien"] == "maison").groupby(echantillon["etiquette_dpe"], observed=True).mean() * 100

tableau_dpe = (
    echantillon.groupby("etiquette_dpe", observed=True)
    .agg(
        nombre_ventes=("id_mutation", "size"),
        prix_m2_moyen=("prix_m2", "mean"),
        prix_m2_median=("prix_m2", "median"),
        surface_mediane=("surface", "median"),
    )
)
tableau_dpe["part_maisons_pct"] = part_maisons
tableau_dpe.round(1)

In [ ]:
ORDRE_PERIODES = ["avant 1948", "1948-1974", "1975-1977", "1978-1982", "1983-1988",
                  "1989-2000", "2001-2005", "2006-2012", "2013-2021", "après 2021"]

# Répartition des périodes de construction à l'intérieur de chaque étiquette (en %)
tableau_construction = (
    pd.crosstab(echantillon["periode_construction"], echantillon["etiquette_dpe"], normalize="columns")
    .reindex(ORDRE_PERIODES)
    .mul(100)
)
tableau_construction.round(1)

Le prix médian au m² diminue de la classe A (3 204 €) à la classe F (2 319 €), la classe G restant proche de F (2 347 €). Le prix moyen, lui, varie peu d'une classe à l'autre : quelques ventes très chères, surtout de petits appartements dans les grandes villes, le tirent vers le haut dans les classes F et G. La part de maisons diffère aussi beaucoup selon l'étiquette : 83 % des ventes classées A, 39 % des C, mais 65 % des G.

Les périodes de construction déclarées diffèrent fortement selon la classe. 97,2 % des ventes classées G et 93,3 % de celles classées F concernent des logements construits avant 1975, contre 6,8 % des ventes classées A, dont 68,8 % datent de 2013 à 2021. La période de construction est donc une caractéristique à prendre en compte pour comparer les logements entre classes. Ces tableaux décrivent une association et ne permettent pas d'isoler l'effet de l'ancienneté.

In [ ]:
# Carte de la part de logements classés F ou G parmi les ventes appariées, par département
part_fg_dep = (echantillon["etiquette_dpe"].isin(["F", "G"])
               .groupby(echantillon["code_departement"], observed=True).mean()
               .mul(100)
               .reset_index(name="part_fg"))
part_fg_dep["code_departement"] = part_fg_dep["code_departement"].astype(str)
carte_fg = departements.merge(part_fg_dep, left_on="code", right_on="code_departement", how="left")

CLASSES_FG = [0, 5, 10, 15, 20, 100]
NOMS_FG = ["moins de 5 %", "5 à 10 %", "10 à 15 %", "15 à 20 %", "plus de 20 %"]
carte_fg["classe_fg"] = pd.cut(carte_fg["part_fg"], CLASSES_FG, labels=NOMS_FG)

oranges = ListedColormap(plt.cm.Oranges(np.linspace(0.25, 1, len(NOMS_FG))))
fig, ax = plt.subplots(figsize=(7, 7))
carte_fg.plot(column="classe_fg", cmap=oranges, categorical=True, legend=True, ax=ax,
              edgecolor="white", linewidth=0.3,
              missing_kwds={"color": "lightgrey", "label": "hors champ"},
              legend_kwds={"title": "Ventes F ou G", "loc": "center left",
                           "bbox_to_anchor": (1, 0.5), "frameon": False})
ax.set_title("Part des logements classés F ou G parmi les ventes appariées, 2022 à 2025")
ax.set_axis_off()
plt.show()

La part de logements classés F ou G varie fortement d'un département à l'autre : de 2,0 % dans l'Aude à 40,0 % dans la Creuse. Les parts les plus élevées se trouvent dans des départements ruraux du centre de la France (Creuse, Cantal, Nièvre, Lozère, Orne), où le climat est plus froid et le bâti plus ancien. Les plus faibles se trouvent sur le pourtour méditerranéen et en Corse. Comme les prix varient aussi selon les territoires, la comparaison entre étiquettes doit se faire à l'intérieur d'une même commune, ce que fait la section 2.3.

In [ ]:
RANG_DPE = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}

donnees_correlation = echantillon[["prix_m2", "surface", "conso_5_usages_par_m2_ep"]].copy()
donnees_correlation["rang_dpe"] = echantillon["etiquette_dpe"].map(RANG_DPE).astype(float)

# Ancienneté du bâti : seulement pour les années cohérentes avec la période déclarée
annee = pd.to_numeric(echantillon["annee_construction"], errors="coerce")
annee_vente = echantillon["date_mutation"].dt.year
bornes = [float("-inf"), 1947, 1974, 1977, 1982, 1988, 2000, 2005, 2012, 2021, float("inf")]
periode_calculee = pd.cut(annee, bins=bornes, labels=ORDRE_PERIODES)
coherence_periode = (periode_calculee.astype("string")
                     .eq(echantillon["periode_construction"].astype("string"))
                     .fillna(False))
age_valide = coherence_periode & annee.gt(0) & annee.mod(1).eq(0) & annee.le(annee_vente)
donnees_correlation["age_bati"] = (annee_vente - annee).where(age_valide)

print("Années incohérentes avec la période déclarée :", int((annee.notna() & ~coherence_periode).sum()))
print("Ancienneté exploitable :", int(age_valide.sum()), "ventes sur", len(echantillon))

correlations = donnees_correlation.corr(method="spearman")
correlations.round(3)

In [ ]:
libelles = ["Prix au m²", "Surface", "Consommation", "Rang DPE", "Ancienneté déclarée"]

fig_corr, ax = plt.subplots(figsize=(8, 7))
image_corr = ax.imshow(correlations, cmap="RdBu_r", vmin=-1, vmax=1)
for i in range(len(correlations)):
    for j in range(len(correlations)):
        valeur = correlations.iloc[i, j]
        ax.text(j, i, f"{valeur:.3f}", ha="center", va="center",
                color="white" if abs(valeur) >= 0.6 else "black", fontsize=11)
ax.set_xticks(range(5), libelles, rotation=35, ha="right")
ax.set_yticks(range(5), libelles)
ax.set_title("Corrélations de Spearman, 2022 à 2025", pad=16)
fig_corr.colorbar(image_corr, ax=ax, label="Coefficient de Spearman")

n_total = len(donnees_correlation)
n_age = donnees_correlation["age_bati"].notna().sum()
fig_corr.text(0.5, 0.02,
              f"Quatre premières variables : {n_total:,} ventes ; avec l'ancienneté : {n_age:,} ventes.\n"
              "A = 1, G = 7. Ancienneté calculée à la vente sur les années cohérentes.",
              ha="center", fontsize=9)
fig_corr.tight_layout(rect=[0, 0.1, 1, 1])
fig_corr.savefig(config.REPORTS / "figures" / "correlations_spearman.png", dpi=200, bbox_inches="tight")
plt.show()

Le rang DPE et la consommation sont très fortement liés (Spearman : 0,92), ce qui est attendu puisque l'étiquette est attribuée à partir de la consommation. L'ancienneté déclarée est associée positivement à la consommation et au rang DPE (0,56 dans les deux cas) : les logements plus anciens tendent à consommer plus et à être moins bien classés.

Les associations avec le prix au m² restent faibles : -0,13 avec le rang DPE et -0,17 avec l'ancienneté. Ces corrélations ne tiennent compte ni de la localisation, ni de la surface, ni du type de bien.

Les corrélations impliquant l'ancienneté portent sur 937 598 ventes, contre 1 568 214 pour les autres paires : l'année de construction manque souvent, et 32 529 années sont incohérentes avec la période déclarée. Le sous-échantillon disposant d'une ancienneté exploitable peut différer de l'échantillon complet.

### 2.2 Comparaison sans correction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, type_bien in zip(axes, ["appartement", "maison"]):
    selection = echantillon[echantillon["type_bien"] == type_bien]
    groupes = [selection.loc[selection["etiquette_dpe"] == e, "prix_m2"] for e in ORDRE]
    boites = ax.boxplot(groupes, showfliers=False, patch_artist=True)
    for boite, e in zip(boites["boxes"], ORDRE):
        boite.set_facecolor(COULEURS[e])
    ax.set_xticks(range(1, len(ORDRE) + 1), ORDRE)
    ax.set_title(type_bien.capitalize() + "s")
    ax.set_xlabel("Étiquette DPE")
axes[0].set_ylabel("Prix de vente (€/m²)")
axes[0].set_ylim(bottom=0)
fig.suptitle("Prix au m² selon l'étiquette, 2022 à 2025")
fig.text(0.5, 0.02, "Points au-delà des moustaches masqués ; toutes les valeurs participent aux calculs.",
         ha="center", fontsize=9)
fig.tight_layout(rect=[0, 0.07, 1, 0.94])
fig.savefig(config.REPORTS / "figures" / "boxplots_dpe.png", dpi=200)
plt.show()

Pour les maisons, le prix médian au m² diminue régulièrement de la classe A (3 056 €) à la classe G (1 760 €). Pour les appartements, la tendance s'inverse à partir de la classe E : les appartements classés G ont le prix médian le plus élevé (4 998 €/m²). Ce sont surtout de petits logements, 34 m² en médiane contre 61 m² pour la classe A, souvent situés dans les centres des grandes villes, où le prix au m² est élevé. Les distributions de prix se recouvrent d'ailleurs largement d'une classe à l'autre.

Cette comparaison brute mélange donc l'effet de l'étiquette avec celui de la localisation, de la surface et du type de bien. C'est pour cela que la section 2.3 compare des logements semblables.

### 2.3 Résultat principal : l'écart à logement comparable

Cette partie présente les différences de prix au mètre carré observées entre les logements selon leur étiquette énergétique. Deux références sont utilisées : la classe G, qui correspond à la moins bonne performance énergétique, et la classe D, située au milieu de l'échelle.

Il faut distinguer deux mesures. Le premier écart compare directement les prix moyens des logements de deux classes. Il ne tient donc pas compte des différences de composition entre les groupes. Le second écart est calculé sur des groupes de logements comparables, constitués à partir de la commune, du type de bien, de la tranche de surface, de la période de construction et du trimestre de vente. Cette seconde mesure permet de limiter l'influence de ces caractéristiques sur la comparaison.

Les intervalles de confiance à 95 % sont obtenus par bootstrap, en rééchantillonnant les communes. Cette méthode tient compte du fait que les observations d'une même commune peuvent être liées entre elles. Les estimations présentées reposent sur 1 000 répétitions.

In [ ]:
from src import produire_resultats, produire_robustesse, produire_evolution_annuelle
from src import visualiser_resultats, visualiser_evolution_annuelle

# Écarts bruts et comparables, références G et D, intervalles par bootstrap (1 000 répétitions)
resultats = produire_resultats.calculer_resultats(ventes)


def tableau_ecarts(resultats, reference):
    t = resultats[resultats["reference"] == reference].set_index("etiquette")
    return pd.DataFrame({
        "écart brut (%)": t["ecart_brut_pct"],
        "écart à logement comparable (%)": t["ecart_comparable_pct"],
        "IC 95 % bas": t["ic_comparable_inf"],
        "IC 95 % haut": t["ic_comparable_sup"],
        "part de la classe couverte (%)": t["part_cible_support_pct"],
    }).round(2)

#### 2.3.1 Comparaison avec la classe G

In [ ]:
tableau_ecarts(resultats, "G")

Les écarts sont positifs pour toutes les classes comparées à G. Autrement dit, dans notre échantillon, les logements ayant une meilleure étiquette énergétique se vendent en moyenne plus cher que les logements classés G. L'écart est particulièrement marqué pour les classes A, B et C.

La comparaison entre logements similaires donne également des écarts positifs. Elle montre notamment que la différence entre F et G, peu visible dans la comparaison brute, atteint 2,54 % après prise en compte des caractéristiques retenues pour former les groupes comparables.

Il faut toutefois rester prudent pour les classes A et B. Seules 0,42 % des ventes classées A et 2,04 % des ventes classées B se trouvent dans un groupe qui contient aussi des logements classés G (colonne « part de la classe couverte »). Ces comparaisons reposent donc sur peu de ventes et sont plus sensibles à la composition de l'échantillon.

#### 2.3.2 Comparaison avec la classe D

In [ ]:
tableau_ecarts(resultats, "D")

La classe D permet d'observer les écarts de part et d'autre d'une étiquette intermédiaire. Les classes A, B et C présentent des prix plus élevés que D, tandis que les classes E, F et G présentent des prix plus faibles.

Après comparaison de logements similaires, l'écart atteint -8,72 % pour F et -11,00 % pour G par rapport à D. Les intervalles de confiance correspondants ne contiennent pas zéro. Dans le cadre de cette méthode, les écarts estimés sont donc relativement précis.

La comparaison brute et la comparaison entre logements similaires ne donnent pas toujours le même ordre de grandeur. C'est notamment le cas de la classe C, pour laquelle l'écart avec D passe de 10,54 % à 3,94 %. Cela montre l'importance de tenir compte des caractéristiques des logements et de leur localisation avant d'interpréter les différences de prix.

In [ ]:
fig = visualiser_resultats.graphique_ecarts(resultats)
plt.show()

Dans l'ensemble, les résultats font apparaître une association entre la performance énergétique et le prix de vente. Ils ne permettent cependant pas d'affirmer que l'amélioration de l'étiquette énergétique entraîne, à elle seule, une hausse du prix. D'autres caractéristiques, qui ne sont pas toutes observées dans les données, peuvent également expliquer une partie des différences.

### 2.4 Évolution des écarts de 2022 à 2025

Les résultats précédents portent sur l'ensemble de la période étudiée. Il est également intéressant de vérifier si les écarts entre les logements classés F ou G et ceux classés D évoluent d'une année à l'autre.

Pour cette analyse, la classe D est conservée comme référence. Les écarts sont calculés séparément pour chaque année, selon la même logique de comparaison entre logements similaires.

In [ ]:
evolution = produire_evolution_annuelle.calculer_evolution(ventes)
evolution[["annee", "etiquette", "ecart_brut_pct", "ecart_comparable_pct", "part_cible_support_pct"]].round(2)

In [ ]:
fig = visualiser_evolution_annuelle.graphique_evolution(evolution)
plt.show()

Les résultats montrent que les logements classés F et G se vendent moins cher que ceux de classe D sur chacune des années observées. L'écart comparable devient globalement plus important au cours de la période.

Pour les logements classés F, il passe de -6,72 % en 2022 à -10,54 % en 2025. Pour les logements classés G, il évolue de -8,07 % à -13,18 %. Cette évolution est particulièrement visible pour la classe G.

Cette période correspond également à plusieurs changements réglementaires concernant les logements les moins performants sur le plan énergétique :

- **24 août 2022 :** entrée en vigueur du gel des augmentations de loyers pour les logements classés F et G en France métropolitaine, pour les contrats concernés.
- **1er janvier 2023 :** le seuil de 450 kWh d'énergie finale par mètre carré et par an entre dans le critère de décence énergétique. Les logements qui dépassent ce seuil ne satisfont plus ce critère.
- **1er janvier 2025 :** les logements classés G sont concernés par l'interdiction de location au titre du calendrier de décence énergétique.

Ces dates sont reportées sur le graphique afin de situer l'évolution des écarts dans leur contexte réglementaire. Elles ne permettent toutefois pas d'attribuer directement les variations observées à ces mesures. D'autres facteurs ont pu influencer le marché immobilier pendant la même période.

Enfin, la part de logements de la classe cible couverte par les comparaisons augmente au fil des années. Les résultats annuels doivent donc être interprétés en tenant compte de cette évolution de la couverture. Les intervalles de confiance présentés dans la section précédente ne sont pas recalculés ici pour chaque année.

## Partie 3. Vérification et limites

### 3.1 Appariement strict

Les résultats peuvent dépendre de la fiabilité de l'appariement entre les ventes et les diagnostics. Pour vérifier ce point, une seconde analyse ne garde que les appariements les plus sûrs : elle retire les ventes marquées comme ambiguës et celles appariées par la distance plutôt que par l'adresse.

Les estimations obtenues avec cette méthode sont comparées à celles de l'analyse principale. Les intervalles de confiance sont également présentés pour les deux méthodes.

In [ ]:
resultats_stricts = produire_resultats.calculer_resultats(ventes, strict=True)
comparaison_strict = produire_robustesse.comparer_standard_strict(resultats, resultats_stricts)
comparaison_strict[["reference", "etiquette",
                    "ecart_comparable_pct_standard", "ic_comparable_inf_standard", "ic_comparable_sup_standard",
                    "ecart_comparable_pct_strict", "ic_comparable_inf_strict", "ic_comparable_sup_strict"]].round(2)

Les estimations obtenues avec l'appariement strict sont légèrement plus élevées en valeur absolue que celles de l'analyse principale. C'est le cas, par exemple, de l'écart entre les classes G et D, qui passe de -11,00 % à -11,70 %.

Malgré ces différences, les résultats conservent le même sens : les logements mieux classés présentent des prix plus élevés que ceux des classes énergétiques moins performantes, dans les comparaisons effectuées. Les intervalles de confiance restent également du même côté de zéro.

Cette vérification est plutôt rassurante : les principaux constats ne reposent pas uniquement sur le choix des critères d'appariement de l'analyse initiale. Il faut néanmoins noter que les critères plus stricts réduisent le nombre de logements pouvant être comparés. La couverture de l'échantillon est donc un peu plus faible.

### 3.2 Limites de l'étude

Plusieurs limites doivent être prises en compte pour interpréter les résultats.

**La qualité de l'appariement.** Le rapprochement entre les données de ventes et les diagnostics énergétiques repose sur des informations de localisation et sur les caractéristiques des biens. Malgré les contrôles effectués, certaines correspondances peuvent rester incertaines. Ce problème peut notamment se poser dans les immeubles comportant plusieurs logements à une même adresse.

**La sélection des logements observés.** L'étude porte sur des logements vendus et pour lesquels un diagnostic énergétique a pu être associé. Elle ne couvre donc pas l'ensemble du parc immobilier français. Les logements qui ne sont pas vendus pendant la période étudiée ne figurent pas dans l'échantillon.

**Les caractéristiques non observées.** La comparaison tient compte de plusieurs éléments, comme la commune, le type de bien, la surface, la période de construction et le trimestre de vente. D'autres facteurs peuvent néanmoins influencer le prix : l'état général du logement, sa localisation précise, la présence d'un extérieur, l'étage, la qualité des équipements ou encore les travaux réalisés.

**La représentation inégale des classes énergétiques.** Certaines classes, en particulier A et B dans la comparaison avec G, sont peu représentées. Les résultats qui les concernent doivent donc être lus avec davantage de prudence.

**L'interprétation des écarts.** Les méthodes utilisées permettent de comparer les prix observés entre des groupes de logements présentant des caractéristiques proches. Elles ne suffisent pas à établir un lien de causalité. Il n'est donc pas possible d'affirmer que la seule amélioration du DPE entraîne la hausse du prix de vente.

Enfin, le bootstrap par commune permet de tenir compte de la dépendance possible entre les observations d'une même commune pour le calcul des intervalles de confiance. Il ne corrige cependant pas les autres limites liées à la qualité des données ou aux caractéristiques qui ne sont pas observées.

## Conclusion

Cette étude avait pour objectif d'examiner le lien entre la performance énergétique des logements et leur prix de vente en France. Pour cela, les données de ventes immobilières ont été rapprochées des diagnostics de performance énergétique, puis les prix ont été comparés selon les différentes étiquettes du DPE.

Les résultats font apparaître des écarts de prix selon la classe énergétique. Dans la comparaison entre logements similaires, les biens classés F se vendent en moyenne 8,72 % moins cher que ceux classés D, tandis que l'écart atteint 11,00 % pour les logements classés G. Les comparaisons avec la classe G montrent également des écarts positifs pour les classes énergétiques supérieures.

L'analyse annuelle indique que les écarts entre les classes F et G et la classe D deviennent globalement plus importants entre 2022 et 2025. La vérification avec des critères d'appariement plus stricts conduit à des résultats proches de ceux de l'analyse principale. Les principaux constats semblent donc relativement stables face à cette modification de la méthode.

Ces résultats doivent toutefois être interprétés avec prudence. Les données disponibles ne permettent pas de tenir compte de toutes les caractéristiques susceptibles d'influencer le prix d'un logement. De plus, l'étude repose sur les biens vendus et correctement rapprochés avec un diagnostic énergétique. Elle met donc en évidence une association entre l'étiquette énergétique et le prix, mais ne permet pas de conclure à un effet causal du DPE sur la valeur immobilière.

Malgré ces limites, les résultats suggèrent que la performance énergétique constitue un élément à prendre en considération dans l'analyse du marché immobilier. Les écarts observés entre les classes D, F et G sont cohérents avec l'idée que les logements les moins performants peuvent faire l'objet d'une décote sur le marché. Cette question mérite une attention particulière dans un contexte où les exigences réglementaires relatives à la location des logements énergivores se renforcent.



## Annexes

### Annexe A. Règles d'appariement

Une vente est rattachée à un DPE si les six règles suivantes sont respectées. Les seuils sont définis dans `src/config.py`.

| Règle | Contenu | Seuil |
|---|---|---|
| 1. Même adresse et même type de logement | Même commune, même voie et même numéro, après avoir écrit les adresses des deux bases de la même façon | |
| 2. À défaut, un DPE tout proche | Quand l'adresse ne permet pas de conclure, DPE situé à faible distance de la vente | 20 m pour un appartement, 60 m pour une maison |
| 3. Une surface proche | Écart entre la surface de la vente (DVF) et la surface habitable du DPE | au plus 10 % pour un appartement, 20 % pour une maison |
| 4. Un DPE établi avant la vente | DPE antérieur à la vente ; son ancienneté entre dans le score, si bien qu'à surface égale le plus récent est préféré | au plus 3 ans avant la vente |
| 5. Un DPE pour un seul logement | Un même DPE ne peut pas être attribué à deux logements différents ; il revient à celui qui lui correspond le mieux. Plusieurs ventes d'un même logement peuvent en revanche partager son DPE | |
| 6. Les cas ambigus sont signalés | Vente marquée « ambiguë » quand un autre DPE d'étiquette différente convient presque aussi bien | écart de score inférieur à 0,10 |

Quand plusieurs DPE respectent ces règles, on retient celui dont le score est le plus faible :

score = écart de surface / tolérance + 0,25 × ancienneté du DPE / 3 ans + distance / rayon (la distance ne compte que pour la règle 2).

Sont aussi écartés : les DPE dont l'adresse a été mal reconnue par la Base Adresse Nationale (note de fiabilité inférieure à 0,5), les DPE remplacés par un DPE plus récent et les DPE qui ne portent pas sur une maison ou un appartement (DPE d'immeuble entier). Les ventes d'un même logement à moins de 30 jours d'écart sont signalées et exclues de l'échantillon d'analyse.

### Annexe B. Taux d'appariement par département

Le taux par année et par type de bien est donné en partie 1.3.

In [ ]:
taux_dep = (ventes.groupby("code_departement", observed=True)["apparie"]
            .mean()
            .mul(100)
            .reset_index(name="taux"))
taux_dep["code_departement"] = taux_dep["code_departement"].astype(str)
carte_taux = departements.merge(taux_dep, left_on="code", right_on="code_departement", how="left")

CLASSES_TAUX = [0, 30, 40, 50, 60, 100]
NOMS_TAUX = ["moins de 30 %", "30 à 40 %", "40 à 50 %", "50 à 60 %", "plus de 60 %"]
carte_taux["classe_taux"] = pd.cut(carte_taux["taux"], CLASSES_TAUX, labels=NOMS_TAUX)

bleus_taux = ListedColormap(plt.cm.Blues(np.linspace(0.25, 1, len(NOMS_TAUX))))
fig, ax = plt.subplots(figsize=(7, 7))
carte_taux.plot(column="classe_taux", cmap=bleus_taux, categorical=True, legend=True, ax=ax,
                edgecolor="white", linewidth=0.3,
                missing_kwds={"color": "lightgrey", "label": "hors champ"},
                legend_kwds={"title": "Ventes appariées", "loc": "center left",
                             "bbox_to_anchor": (1, 0.5), "frameon": False})
ax.set_title("Part des ventes appariées à un DPE, 2022 à 2025")
ax.set_axis_off()
plt.show()

In [ ]:
taux_dep_annee = (ventes.groupby(["code_departement", "annee"], observed=True)["apparie"]
                  .mean()
                  .mul(100)
                  .unstack()
                  .round(1))
taux_dep_annee["ensemble"] = ventes.groupby("code_departement", observed=True)["apparie"].mean().mul(100).round(1)
taux_dep_annee.sort_values("ensemble")

### Annexe C. Ventes appariées et non appariées

Comparaison sur les variables connues pour toutes les ventes, par type de bien, sur l'ensemble de la période.

In [ ]:
statut = ventes["apparie"].map({True: "appariées", False: "non appariées"})

comparaison_detail = (ventes.groupby(["type_bien", statut], observed=True)
                      .agg(ventes=("prix_m2", "size"),
                           prix_m2_q1=("prix_m2", lambda x: x.quantile(0.25)),
                           prix_m2_median=("prix_m2", "median"),
                           prix_m2_q3=("prix_m2", lambda x: x.quantile(0.75)),
                           surface_mediane=("surface", "median"),
                           pieces_mediane=("nb_pieces", "median"))
                      .round(1))
comparaison_detail.index.names = ["type de bien", "statut"]
comparaison_detail

Répartition des étiquettes dans l'échantillon d'analyse, comparée à celle des résidences principales au 1er janvier 2024 (SDES, 2024). Le parc ne tient pas compte du changement des seuils pour les logements de moins de 40 m² entré en vigueur le 1er juillet 2024.

In [ ]:
# Résidences principales au 1er janvier 2024 (SDES, 2024), en %
PARC_2024 = pd.Series({"A": 2.6, "B": 4.6, "C": 25.7, "D": 32.1, "E": 21.1, "F": 8.1, "G": 5.7})

repartition = pd.DataFrame({
    "échantillon": echantillon["etiquette_dpe"].value_counts(normalize=True).reindex(ORDRE) * 100,
    "parc 2024": PARC_2024,
}).round(1)
display(repartition)

x = np.arange(len(ORDRE))
largeur = 0.4
fig, ax = plt.subplots()
ax.bar(x - largeur / 2, repartition["échantillon"], largeur, color=[COULEURS[e] for e in ORDRE], label="ventes appariées")
ax.bar(x + largeur / 2, repartition["parc 2024"], largeur, color="lightgrey", label="résidences principales (2024)")
ax.set_xticks(x, ORDRE)
ax.set(ylabel="%", title="Répartition des étiquettes : ventes appariées et parc de logements")
ax.legend(frameon=False)
plt.show()

### Annexe D. Dictionnaire des variables

Variables chargées par `chargement.charger_ventes()`. Une ligne correspond à une vente.

**Vente (source : DVF)**

| Variable | Description |
|---|---|
| `id_mutation` | Identifiant de la vente dans DVF |
| `date_mutation` | Date de la vente |
| `annee`, `trimestre` | Année et trimestre de la vente, calculés à partir de la date |
| `prix` | Prix de vente, en euros |
| `type_bien` | Maison ou appartement |
| `surface` | Surface du logement en m² : surface « loi Carrez » pour un appartement quand elle est renseignée, sinon surface bâtie |
| `nb_pieces` | Nombre de pièces principales |
| `prix_m2` | Prix divisé par la surface, en euros par m² |
| `code_commune`, `code_departement` | Code Insee de la commune et code du département |
| `longitude`, `latitude` | Coordonnées du centre de la parcelle cadastrale |

**Appariement (calculé par la chaîne de traitement)**

| Variable | Description |
|---|---|
| `apparie` | Vrai si un DPE a été rattaché à la vente |
| `vente_rapprochee` | Vrai si le même logement a été vendu une autre fois à moins de 30 jours d'écart (exclue de l'échantillon d'analyse) |
| `methode_appariement` | « adresse » (règle 1) ou « spatial » (règle 2, DPE proche) |
| `ambigu` | Vrai si un autre DPE d'étiquette différente convenait presque aussi bien (règle 6) |
| `n_candidats` | Nombre de DPE qui respectaient les règles pour cette vente |
| `delai_jours` | Nombre de jours entre l'établissement du DPE et la vente |
| `ecart_surface` | Écart relatif entre la surface de la vente et la surface habitable du DPE (0,05 = 5 %) |
| `distance_m` | Distance en mètres entre la vente et le DPE, pour les appariements par proximité |

**Diagnostic (source : base des DPE de l'ADEME, renseigné pour les ventes appariées)**

| Variable | Description |
|---|---|
| `etiquette_dpe` | Étiquette énergie, de A (la meilleure) à G |
| `etiquette_ges` | Étiquette climat (émissions de gaz à effet de serre), de A à G |
| `conso_5_usages_par_m2_ep` | Consommation conventionnelle d'énergie primaire, en kWh par m² et par an, pour cinq usages : chauffage, eau chaude, refroidissement, éclairage et auxiliaires |
| `version_dpe` | Version de la méthode de calcul du DPE |
| `periode_construction` | Période de construction du logement |
| `annee_construction` | Année de construction, quand elle est connue |
| `zone_climatique` | Zone climatique réglementaire du logement (H1, H2 ou H3, avec leurs sous-zones) |

## Références

*À compléter à partir de la note de projet.*